## TUGAS MANDIRI - Debi Aprilia - 2320506038

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 7 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Tim data engineering platform e-commerce meminta anda membangun **pipeline ETL produksi pertama** yang menggabungkan tiga sumber data sekaligus: transaksi (CSV), data produk (JSON), dan data ulasan pelanggan (CSV terpisah) — merepresentasikan kondisi nyata di mana data tersebar di berbagai sistem dengan format berbeda-beda.

### Menyiapkan Dataset

Jalankan cell berikut untuk menghasilkan **tiga sumber data** sekaligus.

In [2]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


### Instruksi Pengerjaan

Buat notebook baru **`Tugas6_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession`, lalu bangun pipeline ETL lengkap mengikuti struktur **Extract → Transform → Load**:

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, when, round

spark = SparkSession.builder \
    .appName("Tugas6_ETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/09/28 22:55:02 WARN Utils: Your hostname, deb-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/09/28 22:55:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/28 22:55:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


**A. EXTRACT** *(bobot 15%)*

Baca ketiga sumber data (`tugas6_transaksi.csv`, `tugas6_produk.json`, `tugas6_ulasan.csv`) menjadi tiga Spark DataFrame terpisah. Tampilkan jumlah baris dan `printSchema()` masing-masing.

In [3]:
#Membaca ketiga sumber data menjadi tiga spark DataFrame terpisah
df_transaksi = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk = spark.read.json("tugas6_produk.json")
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

#Menampilkan jumlah baris dengan perintah count() dan menampilkan skema masing-masing DataFrame
print(f"Jumlah transaksi: {df_transaksi.count()} baris")
df_transaksi.printSchema()

print(f"Jumlah produk: {df_produk.count()} baris")
df_produk.printSchema()

print(f"Jumlah ulasan: {df_ulasan.count()} baris")
df_ulasan.printSchema()

Jumlah transaksi: 5000 baris
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

Jumlah produk: 30 baris
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

Jumlah ulasan: 3500 baris
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



**B. TRANSFORM — Penggabungan** *(bobot 25%)*

Gabungkan ketiga DataFrame menjadi satu (`order_id` sebagai kunci ke ulasan, `product_id` sebagai kunci ke produk). Gunakan **`salah satu join yang tepat`** dari transaksi ke ulasan (karena tidak semua transaksi memiliki ulasan) dan **`salah satu join yang tepat`** dari transaksi ke produk (karena setiap transaksi pasti memiliki produk yang valid). Tambahkan kolom `total_pendapatan` (`unit_terjual x harga`).

In [4]:
#Melakukan left join dari transaksi ke ulasan karena tidak semua transaksi punya ulasan
df_merged = df_transaksi.join(df_ulasan, on="order_id", how="left")

#Melakukan inner join dari transaksi ke produk karena setiap transaksi pasti memiliki produk yang valid
df_merged = df_merged.join(df_produk, on="product_id", how="inner")

#Menambahkan kolom total_pendapatan
df_etl = df_merged .withColumn("total_pendapatan", col("unit_terjual") * col("harga"))
df_etl.show(5)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|         1500000|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|          450000|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
o

**C. TRANSFORM — Penanganan Data Kosong & Pengayaan** *(bobot 20%)*

- Transaksi tanpa ulasan akan memiliki `rating` bernilai kosong (`null`) setelah `salah satu join yang tepat` — isi nilai kosong tersebut dengan angka **0** menggunakan `salah satu function`, sertakan alasan singkat mengapa 0 (bukan nilai lain) masuk akal untuk kasus "belum ada ulasan".
- Tambahkan kolom `ada_ulasan` bernilai `True`/`False` (tidak boleh diisi manual satu satu)`, **sebelum** langkah `na.fill()` di atas).


In [5]:
#Membuat kolom ada_ulasan sebelum mengisi nilai null pada rating
df_etl = df_etl.withColumn("ada_ulasan", when(col("rating").isNotNull(), True).otherwise(False))

#Mengisi nilai null dengan angka 0 menggunakan na.fill()
#Alasan mengapa diisi angka 0 : karena untuk rating angka 0 bersifat netral atau bisa menandakan bahwa transaksi belum memiliki ulasan.
#Sehingga tidak akan merusak rata-rata rating asli dari pelanggan yang memberikan ulasan
df_etl = df_etl.na.fill({"rating": 0})

#Menampilkan beberapa baris untuk verifikasi
df_etl.select("order_id", "product_id", "kategori", "rating", "ada_ulasan", "total_pendapatan").show(5)

+--------+----------+------------+------+----------+----------------+
|order_id|product_id|    kategori|rating|ada_ulasan|total_pendapatan|
+--------+----------+------------+------+----------+----------------+
|     TX0|        21|  Elektronik|     1|      true|           50000|
|     TX1|         8|     Fashion|     0|     false|         1500000|
|     TX2|        25|  Elektronik|     2|      true|          100000|
|     TX3|         3|     Fashion|     5|      true|          300000|
|     TX4|        19|Rumah Tangga|     0|     false|          450000|
+--------+----------+------------+------+----------+----------------+
only showing top 5 rows



**D. LOAD** *(bobot 25%)*

Simpan hasil akhir ke HDFS dalam format **Parquet**, dipartisi berdasarkan `kategori`, ke path `/user/[username]/tugas6/hasil_etl`. Verifikasi dengan `hdfs dfs -ls -R`, lalu baca kembali dan tampilkan `count()`-nya sebagai bukti data tersimpan utuh.

In [6]:
#Menentukan path hdfs untuk menyimpan hasil
hdfs_path = "/user/deb/tugas6/hasil_etl"

#Menyimpan ke HDFS dengan format Parquet dan partisi berdasarkan kategori
df_etl.write.mode("overwrite").partitionBy("kategori").parquet(f"hdfs://localhost:9000{hdfs_path}")
print("Data berhasil disimpan ke HDFS")

#Verifikasi struktur folder
!hdfs dfs -ls -R /user/deb/tugas6/hasil_etl

#Membaca kembali file 
df_verifikasi = spark.read.parquet(f"hdfs://localhost:9000{hdfs_path}")
print(f"Total baris data terverifikasi di HDFS: {df_verifikasi.count()}")

Data berhasil disimpan ke HDFS
-rw-r--r--   3 deb supergroup          0 2026-09-28 22:56 /user/deb/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - deb supergroup          0 2026-09-28 22:56 /user/deb/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 deb supergroup      16366 2026-09-28 22:56 /user/deb/tugas6/hasil_etl/kategori=Elektronik/part-00000-9bd5503c-9c60-485d-beda-203c7fb497f7.c000.snappy.parquet
drwxr-xr-x   - deb supergroup          0 2026-09-28 22:56 /user/deb/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 deb supergroup      11133 2026-09-28 22:56 /user/deb/tugas6/hasil_etl/kategori=Fashion/part-00000-9bd5503c-9c60-485d-beda-203c7fb497f7.c000.snappy.parquet
drwxr-xr-x   - deb supergroup          0 2026-09-28 22:56 /user/deb/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 deb supergroup      10392 2026-09-28 22:56 /user/deb/tugas6/hasil_etl/kategori=Kesehatan/part-00000-9bd5503c-9c60-485d-beda-203c7fb497f7.c000.snappy.parquet
drwxr-xr-x   - deb supergroup          0 2026-09-

Total baris data terverifikasi di HDFS: 5000


**E. Insight Akhir** *(bobot 15%)*

Dari data hasil ETL, tampilkan (menggunakan DataFrame API **atau** Spark SQL, bebas memilih): kategori produk mana yang memiliki **persentase transaksi dengan ulasan** (`ada_ulasan = True`) **paling rendah**? Tulis 2-3 kalimat interpretasi bisnis pada markdown cell: mengapa hal ini mungkin penting diketahui oleh tim marketing?


In [7]:
#Menghitung persentase ulasan per kategori (true dihitung 1, false dihitung 0)
df_insight = df_etl.groupBy("kategori").agg(round(avg(when(col("ada_ulasan") == True, 1).otherwise(0)) * 100, 2).alias("persentase_ulasan")).orderBy("persentase_ulasan")
df_insight.show()

+------------+-----------------+
|    kategori|persentase_ulasan|
+------------+-----------------+
|     Makanan|            68.84|
|   Kesehatan|            68.89|
|     Fashion|            70.14|
|Rumah Tangga|            70.55|
|  Elektronik|            70.66|
+------------+-----------------+



**Interpretasi bisnis :** Kategori dengan presentase transaksi dengan ulasan yang paling rendah ada pada kategori Makanan dengan persentase ulasan 68.84%. Hal ini penting diketahui oleh tim marketing karena rendahnya persentase ulasan menunjukkan bahwa pelanggan tidak berminat untuk memberikan feedback setelah membeli produk, sehingga tim marketing perlu informasi ini untuk membuat promosi atau reward bagi pelanggan yang memberi ulasan pada kategori Makanan. Hal tersebut dilakukan agar meningkatkan partisipasi ulasan yang dapat meningkatkan kepercayaan pada pelanggan baru yang akan membeli Makanan.

## Menutup SparkSession

In [8]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
